Objectif : reduire les données issues de MIRISIM pour faire du RDI (source et reference)

In [1]:
#### code copier coller de HR8799 : a adapter avant d'utiliser pour HD 95086 !



# Packages that allow us to get information about objects:
import asdf
import copy
import os
import shutil
import glob
# Numpy library:
import numpy as np
# For downloading data
import requests
# Astropy tools:
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch
#Set up matplotlib for plotting
import matplotlib.pyplot as plt
import matplotlib as mpl
# parallelisation
#from joblib import Parallel, delayed #besoin d'installer joblib

#solution tempo du notebook MRS
import multiprocessing
multiprocessing.set_start_method('fork')
from multiprocessing import Pool
# Set the maximum number of processes to spawn based on available cores
usage = 'none' # Either 'none' (single thread), 'quarter', 'half', or 'all' available cores
maxp = 1
print('We will use '+str(maxp)+' CPUs in this run')

#Import JWST pipeline-related modules
import jwst
from jwst.pipeline import Detector1Pipeline
from jwst.pipeline import Image2Pipeline
from jwst.pipeline import Coron3Pipeline
from jwst.associations import asn_from_list as afl # Tools for creating association files
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase # Definition of a Lvl2 association file
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base # Definition of a Lvl3 association file
from jwst.associations.lib.member import Member
from jwst.associations.asn_from_list import asn_from_list
from tqdm.notebook import tqdm as tqdm
#Check which version of the pipeline we are running:
print(jwst.__version__)



We will use 1 CPUs in this run
1.12.5


In [2]:
obs_name = 'obs1_COR_F1065C'
path_code    = '/home/cperrot/Work/code/JWST/Target/GTO/'
path_product = '/home/cperrot/Work/projet/JWST/Target/GTO/HD95086/'+obs_name+'/'
path_raw     = '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/'+obs_name+'/'

#if not os.path.exists(path_product+'source/'):
#    os.makedirs(path_product+'source/')
#if not os.path.exists(path_product+'reference/'):
#    os.makedirs(path_product+'reference/')

#path_raw    = '/home/cperrot/Work/code/MIRISIM/corono/20220328_162102_mirisim/det_images/'

path_stage1 = path_product+'stage1-2/'
#path_stage2 = '/home/cperrot/Work/code/MIRISIM/corono/corono_S2/'
#path_stage3 = '/home/cperrot/Work/code/MIRISIM/corono/corono_S3/'

if not os.path.exists(path_stage1):
    os.makedirs(path_stage1)
#if not os.path.exists(path_stage2):
#    os.makedirs(path_stage2)
#if not os.path.exists(path_stage3):
#    os.makedirs(path_stage3)

FUNCTION

In [5]:
def rundet1(filenames):
    det1 = Detector1Pipeline() # Instantiate the pipeline
    det1.output_dir = path_stage1 # Specify where the output should go
    det1.ramp_fit.max_cores = 4  #number of core for the data reduction ramp fit
    det1.ipc.skip = True      #Skip something (what ?)
    det1.jump.rejection_threshold = 5.0   #jump rejection value, 5 is default value
    det1.jump.max_cores = 4  #number of core for the data reduction  jump
    #det1.refpix.skip = True # for simulation only Skip the reference pixel subtraction (as it doesn't interact well with simulated data)
    #det1.rscd.skip = True  # for simulation only
    #det1.reset.skip = True # for simulation only
    det1.save_results = True # Save the final resulting _rate.fits files
    print(filenames)
    det1(filenames) # Run the pipeline on an input list of files
    


    
    
# Define a function that can oversee the multiprocessing
# We'll just pass this a function name and a set of inputs, and it'll figure out the
# multithreading accordingly.
def runmany(step, filenames):
    if __name__ == '__main__':
        p = Pool(maxp)
        res = p.map(step, filenames)
        p.close()
        p.join()




def writel2asn(files, asn_name, prodname,**kwargs):
    
    asn = asn_from_list(files, rule=DMSLevel2bBase, product_name=prodname)
    asn.data['products'] = None
    asn['asn_pool'] = asn_name + '_pool'
    asn['asn_type'] = 'image2'
    for n, sci in enumerate(files):
            asn.new_product('{}_exposure{}'.format(asn_name, str(n+1)))
            sci_member = Member({'expname': sci, 'exptype': 'science'})    
            new_members = asn.current_product['members']
            new_members.append(sci_member)
    # print the association and save to file
    name, ser = asn.dump()
    print(ser)
    asn_file2 = path_stage2+asn_name+'.json'
    with open(asn_file2, 'w') as f:
        f.write(ser)
        


In [7]:
# RUN STAGE 1
 
# SOURCE
sstring = path_raw+'*uncal.fits'
raw_files_source = sorted(glob.glob(sstring))
print(raw_files_source)
#param_file = path_stage1+'miri_detector1_param_file.asdf'
print("Run stage 1 of the pipeline - SOURCE")
print(f'Found {len(raw_files_source)} input files to process')
#print('Found the param file :'+param_file)
runmany(rundet1, raw_files_source)



### faire le déplacement de dossier stage1 -> source_stage_1 et reference_stage_1

['/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs1_COR_F1065C/jw01277001001_04101_00001_mirimage_uncal.fits']
Run stage 1 of the pipeline - SOURCE
Found 1 input files to process


2023-11-13 11:54:49,319 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-11-13 11:54:49,323 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-11-13 11:54:49,324 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-11-13 11:54:49,327 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-11-13 11:54:49,329 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-11-13 11:54:49,330 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-11-13 11:54:49,332 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-11-13 11:54:49,333 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-11-13 11:54:49,335 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-11-13 11:54:49,336 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-11-13 1

/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs1_COR_F1065C/jw01277001001_04101_00001_mirimage_uncal.fits


2023-11-13 11:54:49,465 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs1_COR_F1065C/jw01277001001_04101_00001_mirimage_uncal.fits',).
2023-11-13 11:54:49,477 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/projet/JWST/Target/GTO/HD95086/obs1_COR_F1065C/stage1-2/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'outp

2023-11-13 11:54:51,847 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-13 11:54:51,847 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-13 11:54:51,848 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-13 11:54:51,848 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-13 11:54:51,853 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-13 11:54:53,739 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(16, 500, 224, 288) from jw01277001001_04101_00001_mirimage_uncal.fits>,).
2023-11-13 11:54:53,742 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'out

KeyboardInterrupt: 

Process ForkPoolWorker-6:
Traceback (most recent call last):
  File "/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
  File "/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/multiprocessing/pool.py", line 48, in mapstar
    return list(map(*args))
  File "/tmp/ipykernel_11924/2283347110.py", line 13, in rundet1
    det1(filenames) # Run the pipeline on an input list of files
  File "/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/stpipe/step.py", line 479, in run
    step_result = self.process(*args)
  File "/home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/jwst/pipeline/calwebb_detector1.py

In [13]:
obs_name     = 'obs6_COR_F1065C'
path_raw     = '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/'+obs_name+'/'
path_product = '/home/cperrot/Work/projet/JWST/Target/GTO/HD95086/'+obs_name+'/'
path_stage1  = path_product+'stage1/'
if not os.path.exists(path_stage1):
    os.makedirs(path_stage1)


In [ ]:
# RUN STAGE 1
 
# SOURCE
sstring = path_raw+'*uncal.fits'
raw_files_source = sorted(glob.glob(sstring))
#param_file = path_stage1+'miri_detector1_param_file.asdf'
print("Run stage 1 of the pipeline - SOURCE")
print(f'Found {len(raw_files_source)} input files to process')
#print('Found the param file :'+param_file)
runmany(rundet1, raw_files_source)

Run stage 1 of the pipeline - SOURCE
Found 2 input files to process


2023-03-22 11:30:28,523 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-03-22 11:30:28,526 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-03-22 11:30:28,528 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-03-22 11:30:28,529 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-03-22 11:30:28,531 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-03-22 11:30:28,533 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-03-22 11:30:28,534 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-03-22 11:30:28,536 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-03-22 11:30:28,538 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-03-22 11:30:28,539 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-03-22 1

2023-03-22 11:30:30,702 - stpipe.Detector1Pipeline - INFO - Prefetching reference files for dataset: 'jw01277006001_02101_00001_mirimage_uncal.fits' reftypes = ['dark', 'gain', 'ipc', 'linearity', 'mask', 'persat', 'readnoise', 'saturation', 'superbias', 'trapdensity', 'trappars']
2023-03-22 11:30:31,817 - stpipe.Detector1Pipeline - INFO - Prefetch for DARK reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_dark_0076.fits'.
2023-03-22 11:30:31,819 - stpipe.Detector1Pipeline - INFO - Prefetch for GAIN reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_gain_0008.fits'.
2023-03-22 11:30:31,819 - stpipe.Detector1Pipeline - INFO - Prefetch for IPC reference file is 'N/A'.
2023-03-22 11:30:31,820 - stpipe.Detector1Pipeline - INFO - Prefetch for LINEARITY reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_linearity_0032.fits'.
2023-03-22 11:30:31,821 - stpipe.Detector1Pipeline - INFO - Prefetch for MASK reference file is 

2023-03-22 11:30:43,619 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs6_COR_F1065C'}
2023-03-22 11:30:43,642 - stpipe.Detector1Pipeline.linearity - INFO - Using Linearity reference file /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_linearity_0032.fits
2023-03-22 11:30:46,491 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity done
2023-03-22 11:30:46,592 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd running with args (<RampModel(8, 500, 224, 288) from jw01277006001_02101_00001_mirimage_uncal.fits>,).
2023-03-22 11:30:46,594 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file'

2023-03-22 11:32:04,564 - stpipe.Detector1Pipeline.jump - INFO - From highest outlier, two-point found 50129 pixels with at least one CR from five groups.
2023-03-22 11:32:04,565 - stpipe.Detector1Pipeline.jump - INFO - From highest outlier, two-point found 0 pixels with at least one CR from four groups.
2023-03-22 11:32:04,566 - stpipe.Detector1Pipeline.jump - INFO - From highest outlier, two-point found 1 pixels with at least one CR from three groups.
2023-03-22 11:32:39,225 - stpipe.Detector1Pipeline.jump - INFO - Working on integration 4:
2023-03-22 11:32:42,747 - stpipe.Detector1Pipeline.jump - INFO - From highest outlier, two-point found 49428 pixels with at least one CR from five groups.
2023-03-22 11:32:42,748 - stpipe.Detector1Pipeline.jump - INFO - From highest outlier, two-point found 0 pixels with at least one CR from four groups.
2023-03-22 11:32:42,749 - stpipe.Detector1Pipeline.jump - INFO - From highest outlier, two-point found 2 pixels with at least one CR from three g

2023-03-22 12:43:50,706 - stpipe.Detector1Pipeline.ramp_fit - INFO - Step ramp_fit done
2023-03-22 12:43:50,783 - stpipe.Detector1Pipeline.gain_scale - INFO - Step gain_scale running with args (<ImageModel(224, 288) from jw01277006001_02101_00001_mirimage_uncal.fits>,).
2023-03-22 12:43:50,785 - stpipe.Detector1Pipeline.gain_scale - INFO - Step gain_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'gain_scale', 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs6_COR_F1065C'}
2023-03-22 12:43:50,824 - stpipe.Detector1Pipeline.gain_scale - INFO - GAINFACT not found in gain reference file
2023-03-22 12:43:50,826 - stpipe.Detector1Pipeline.gain_scale - INFO - Step will be skipped
2023-03-22 12:43:50,831 - stpipe.Detector1Pipeline.gain_scale - INFO - Step gain_scale done
2023-03

2023-03-22 12:43:51,216 - stpipe.Detector1Pipeline - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/stdatamodels/validate.py:38: ValidationWarning: While validating meta.dither.primary_type the following error occurred:
'BACKGROUND' is not one of ['FULL', 'FULLBOX', 'INTRAMODULE', 'INTRAMODULEBOX', 'INTRAMODULEX', 'INTRASCA', 'SUBARRAY_DITHER', 'WFSC', 'AMI', 'IMAGING', 'MIMF', 'WFSS', '5-POINT-SMALL-GRID', '9-POINT-SMALL-GRID', 'LFLAT_CORON1065', 'LFLAT_CORON1140', 'LFLAT_CORON1550', 'LFLAT_CORONLYOT', 'PFLAT_CORON1065', 'PFLAT_CORON1140', 'PFLAT_CORON1550', 'PFLAT_CORONLYOT', 'CYCLING', 'REULEAUX', 'SPARSE-CYCLING', '2-POINT', '4-POINT-SETS', '2-POINT-MIRI-F770W-WITH-NIRCAM', '3-POINT-MIRI-F770W-WITH-NIRCAM', '4-POINT-MIRI-F770W-WITH-NIRCAM', '9-POINT-MIRI-F770W-WITH-NIRCAM', '3-POINT-MIRI-F1000W-WITH-NIRCAM', '4-POINT-MIRI-F1000W-WITH-NIRCAM', '2-POINT-MIRI-F1280W-WITH-NIRCAM', '3-POINT-MIRI-F1280W-WITH-NIRCAM', '4-POINT-MIRI-F1280W-WITH-NIRCAM', '9-POINT

2023-03-22 12:44:00,849 - stpipe.Detector1Pipeline.ipc - INFO - Step ipc done
2023-03-22 12:44:00,925 - stpipe.Detector1Pipeline.firstframe - INFO - Step firstframe running with args (<RampModel(8, 500, 224, 288) from jw01277006001_02101_00002_mirimage_uncal.fits>,).
2023-03-22 12:44:00,927 - stpipe.Detector1Pipeline.firstframe - INFO - Step firstframe parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs6_COR_F1065C'}
2023-03-22 12:44:01,414 - stpipe.Detector1Pipeline.firstframe - INFO - Step firstframe done
2023-03-22 12:44:01,485 - stpipe.Detector1Pipeline.lastframe - INFO - Step lastframe running with args (<RampModel(8, 500, 224, 288) from jw01277006001_02101_00002_mirimage_uncal.fits>,).
2023-03-22 12:44:01,487 - stpipe.De

2023-03-22 12:44:05,385 - stpipe.Detector1Pipeline.jump - INFO - CR rejection threshold = 4 sigma
2023-03-22 12:44:05,399 - stpipe.Detector1Pipeline.jump - INFO - Using GAIN reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_gain_0008.fits
2023-03-22 12:44:05,437 - stpipe.Detector1Pipeline.jump - INFO - Using READNOISE reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_readnoise_0085.fits
2023-03-22 12:44:05,920 - stpipe.Detector1Pipeline.jump - INFO - Extracting gain subarray to match science data
2023-03-22 12:44:05,922 - stpipe.Detector1Pipeline.jump - INFO - Extracting readnoise subarray to match science data
2023-03-22 12:44:06,324 - stpipe.Detector1Pipeline.jump - INFO - Executing two-point difference method
2023-03-22 12:44:06,546 - stpipe.Detector1Pipeline.jump - INFO - Working on integration 1:
2023-03-22 12:44:09,383 - stpipe.Detector1Pipeline.jump - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/stcal/jump

2023-03-22 13:48:39,429 - stpipe.Detector1Pipeline.ramp_fit - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/stdatamodels/validate.py:38: ValidationWarning: While validating <CubeModel(8, 224, 288) from jw01277006001_02101_00002_mirimage_uncal.fits> the following error occurred:
'BACKGROUND' is not one of ['FULL', 'FULLBOX', 'INTRAMODULE', 'INTRAMODULEBOX', 'INTRAMODULEX', 'INTRASCA', 'SUBARRAY_DITHER', 'WFSC', 'AMI', 'IMAGING', 'MIMF', 'WFSS', '5-POINT-SMALL-GRID', '9-POINT-SMALL-GRID', 'LFLAT_CORON1065', 'LFLAT_CORON1140', 'LFLAT_CORON1550', 'LFLAT_CORONLYOT', 'PFLAT_CORON1065', 'PFLAT_CORON1140', 'PFLAT_CORON1550', 'PFLAT_CORONLYOT', 'CYCLING', 'REULEAUX', 'SPARSE-CYCLING', '2-POINT', '4-POINT-SETS', '2-POINT-MIRI-F770W-WITH-NIRCAM', '3-POINT-MIRI-F770W-WITH-NIRCAM', '4-POINT-MIRI-F770W-WITH-NIRCAM', '9-POINT-MIRI-F770W-WITH-NIRCAM', '3-POINT-MIRI-F1000W-WITH-NIRCAM', '4-POINT-MIRI-F1000W-WITH-NIRCAM', '2-POINT-MIRI-F1280W-WITH-NIRCAM', '3-POINT-MIRI-F128

In [14]:
print("hello")

hello
ERROR! Session/line number was not unique in database. History logging moved to new session 462
